# A model workflow for MAMMA


In [3]:
import rioxarray as rxr
import xarray as xr
import subprocess
import numpy as np  
import matplotlib.pyplot as plt
import subprocess
from netCDF4 import Dataset
import cartopy.crs as ccrs
import pyproj
import netCDF4
import sys

### In the following sections, we will provide a variety of input information, including topographical, rheological and thermal parameters.

### First, you must enter the run name, along with some time constraints and flags to determine the kind of output they would like to produce.

In [4]:
run_name = ""

ext_flag, lateral_flag, explosive_flag, method_of_moments_flag = 'F', 'F', 'T', 'F' 

### Next, enter parameters describing the geometry of this conduit.

In [5]:
# Left (bottom) of the physical domain.
z0 = 0

# Right (top) of the physical domain.
zn = 5291

# 'fixed' => constant radius
# 'linear' => linear change in radius 
# 'trans1' => two radiuses and an transition zone
# 'trans2' => a zone with high radius with a transition zone
# 'external' => radius profile read from external file
radius_model = 'fixed'

radius_fixed = 30

# Number of control volumes in the computational domain. 
comp_cells = 1000

### Now, please enter the number of crystal and gas phases that will occur.

In [6]:
# Number of gas phases. 
ngas = 1

# Number of crystal phases.
ncry = 1

### Here, describe the steady boundary conditions for the model.

In [7]:
# Inlet temperature
t_in = 1159

# Inlet second phase pressure. 
p1_in = 140000000

# Inlet pressure difference ( p2-p1 ) 
delta_p_in = 0

x_es_dis_in = 4.6e-2

# Outlet pressure. 
p_out = 101300

# Inlet first phase velocity.
u1_in = 1.9

# Residual for the convergence of the shooting method. 
# The solution is accepted if one of these conditions is satisfied: 
eps_conv = 1e-5

#Flag for the shooting technique:
# T => the code search for inlet velociy
# F => the velocity is provided as input
shooting = 'T'

### Below, specify the characteristics of the exolved gas.

In [8]:
# equation of state for gas
# 'ideal' => ideal gas law
# 'VDW' => Van der Waals equation of state
gas_law ="IDEAL"

# Critical gas pressure
pc_g = 22064000

# critical gas temperature
tc_g = 647

# exsolved gas heat capacity at constant volume
cv_g = 1571.0000000000000

# exsolved gas adiabatic exponent
gamma_g = 1.29

# exsolved gas reference density 
rho0_g = .588

# exsolved gas reference temperature 
t0_g = 373

# exsolved gas formation energy 
bar_e_g = 0

# exsolved gas reference entropy
s0_g = 0

# gas viscocity
visc_2 = 1.5e-2

# Exsolved gas volume fraction threshold for lateral degassing. 
alfa_2_lat_thr = 1.1

perm0 = 5.0e-3

### Similarly, specify the characteristics of the _dissolved_ gas here.

In [9]:
# dissolved gas reference density 
rho0_d = 1000

# dissolved gas sound speed at atm conditions 
c0_d = 407.02

# dissolved gas heat capacity at constant volume 
cv_d = 3637.58

# dissolved gas adiabatic exponent
gamma_d = 1.11

# dissolved  gas reference temperature
p0_d = 1e8

# dissolved gas formation energy
bar_e_d = 0

# dissolved gas reference entropy
s0_d = 0

# String for exsolution model:
# 'Henry' => Henry's law;
# 'Zhang' => Zhang model (Zhang, JVGR '99).

exsol_model = "Henry"

# Solubility parameter for Henry's law. 
solub = 4.11

solub_exp = .5

### Below, enter the physical parameters of the crystals in the magma chamber.

In [10]:
# crystals reference density 
rho0_c = 2800

# crystals sound speed at atm conditions 
c0_c = 2000

# crystals heat capacity at constant volume 
cv_c = 360

# crystals adiabatic exponent
gamma_c = 3.4

# crystals reference temperature
p0_c = 2.5e7

# crystals formation energy
bar_e_c = 0

# crystals reference entropy
s0_c = 0

# chamber (equilibrium) crystal volume fraction
beta0 = .4

# maximum crystal volume fraction
beta_max = .4

# Crystallization model
cryst_model = "Vitturi2010"

### To finish out the physical makeup of the magma, specify the following attributes of the melt.

In [11]:
# melt reference density 
rho0_m = 2500

# melt sound speed at atm conditions 
c0_m = 1366

# melt heat capacity at constant volume 
cv_m = 707

# melt adiabatic exponent
gamma_m = 2.08

# melt reference temperature
p0_m = 1.4e7

# melt formation energy
bar_e_m = 0

# melt reference entropy
s0_m = 0

### Below, set the algorithms used to simulate the viscosity of the magma.

In [12]:
# Parameter to select the melt viscosity (bubbles and crystal-free) model:
# 'Hess_and_Dingwell1996'
# 'Romano_et_al2003'
# 'Giordano_et_al2008'
# 'Giordano_et_al2009'
# 'Di_Genova_et_al2013_eqn_3,5'
# 'Di_Genova_et_al2013_eqn_4,5'

visc_melt_model = 'Hess_and_Dingwell1996'

# Parameter to choose the model for the influence of the bubbles on the mixture:
# 'Einstein'
# 'Quane-Russel -> For Campi-Flegrei we use 0.63 as reported in the 2004 Report (Task 2.2)
# 'Eilers' -> Mader et al. 2013
# 'Sibree' -> Mader et al. 2013
# 'Taylor' -> Mader et al. 2013
# 'Mackenzie' -> Mader et al. 2013
# 'DucampRaj' -> Mader et al. 2013
# 'BagdassarovDingwell' -> Mader et al. 2013
# 'Rahaman' -> Mader et al. 2013

bubbles_model = 'none'


# Parameter to choose the model for the influence of crystal on the mixture:
# 'Lejeune_and_Richet1995'
# 'Dingwell1993'
# 'Melnik_and_Sparks1999'
# 'Costa2005'
# 'Melnik_and_Sparks2005'
# 'Vona_et_al2011'
# 'Vona_et_al2011_mod'
# 'Vona_et_al2013_eq19'
# 'Vona_et_al2013_eq20'
# 'Vona_et_al2013_eq21' 
# 'Fixed_value'
theta_model = 'Costa2005'

# Fixed value for the relative viscosity of the crystals. 
theta_fixed = 1

### Now, fill out a few paramters relating to the temperature, fragmentation of the magma, and the gravity constant (for extraplanetary options).

In [13]:
# Flag for isothermal runs:
# isothermal = .TRUE. => constant temperature
# isothermal = .FALSE. => variable temperature (an energy equation is solved)

isothermal = 'T'

# Temperature for isothermal runs. 
fixed_temp = 1159

# Parameter to choose the fragmentation model:

#  fragmentation_model = 1 => volume fraction
#  fragmentation_model = 2 => overpressure
#  fragmentation_model = 3 => strain rate
fragmentation_mode = 1

# Threshold for the fragmentation. 
frag_thr = .8

# Acceleration due to gravity
grav = 9.81

### Now, please speicfy the relaxation parameters of the magma to properly simulate stress.

In [14]:
# drag function model
#  'constant' => drag_funct = 1
#  'eval' => drag_funct = f(q)
#  'darcy' => Darcy formulation from Eq. 16 Degruyter et al. 2012
#  'forchheimer' => Forcheeimer from Eq. 16 Degruyter et al. 2012
#  'drag' => Stokes law for rising bubbles
#  'single_velocity' => Single velocity model (u1=u2)
drag_funct_model = "forchheimer"

LOG10_DRAG_FUNCT_COEFF = 0

# pressure relaxation model
#  'constant' => tau_p = 1
#  'eval' => tau_p = f(q)

p_relax_model = 'constant'

log10_tau_p_coeff = -8

log10_tau_c = -8

log10_tau_d = -8

### Finally, if using the Forchheimer or Darcy drag functions, specify the parameters relating to that below.

In [15]:
log10_bubble_number_density = 15

tortuosity_factor = 3.5

throat_bubble_ratio = 0.1

friction_coefficient = 10.0

c_d = .8

r_a = 1e-3

### Below, the notebook is formatting your input files, and will then be able to run the model! 

In [ ]:
inp = f"""&RUN_PARAMETERS
 RUN_NAME="{run_name}",
 VERBOSE_LEVEL=0          ,
 EXT_WATER_FLAG={ext_flag},
 LATERAL_DEGASSING_FLAG={lateral_flag},
 EXPLOSIVE_FLAG={explosive_flag},
 METHOD_OF_MOMENTS_FLAG={method_of_moments_flag},
 /
&GEOMETRY_PARAMETERS
 Z0=  {z0}     ,
 ZN=  {zn}     ,
 RADIUS_MODEL="{radius_model}",
 RADIUS_FIXED=  {radius_fixed}     ,
 RADIUS_MIN=  0.0000000000000000     ,
 RADIUS_MAX=  0.0000000000000000     ,
 RADIUS_Z=  0.0000000000000000     ,
 RADIUS_Z_SIG=  0.0000000000000000     ,
 ECCEN_FIXED=  0.0000000000000000     ,
 ECCEN_BASE=  0.0000000000000000     ,
 ECCEN_TOP=  0.0000000000000000     ,
 ECCEN_Z_BASE=  0.0000000000000000     ,
 ECCEN_Z_TOP=  0.0000000000000000     ,
 ECCEN_AXIS_B=  0.0000000000000000     ,
 COMP_CELLS= {comp_cells}       ,
 /
&PHASES_PARAMETERS
 N_GAS={ngas}          ,
 N_CRY={ncry}          ,
 /
&STEADY_BOUNDARY_CONDITIONS
 T_IN=  {t_in}     ,
 P1_IN=  {p1_in}     ,
 DELTA_P_IN=  {delta_p_in}     ,
 X_EX_DIS_IN= {x_es_dis_in},
 P_OUT=  {p_out}     ,
 U1_IN=  {u1_in}     ,
 EPS_CONV=  {eps_conv},
 SHOOTING={shooting},
 /
&EXSOLVED_GAS_PARAMETERS
 GAS_LAW="{gas_law}",
 PC_G=  {pc_g}     ,
 TC_G=  {tc_g}     ,
 CV_G=  {cv_g}     ,
 GAMMA_G=  {gamma_g}     ,
 RHO0_G= {rho0_g}     ,
 T0_G=  {t0_g}     ,
 BAR_E_G=  {bar_e_g}     ,
 S0_G= {s0_g}     ,
 VISC_2=  {visc_2},
 ALFA2_LAT_THR=  {alfa_2_lat_thr}     ,
 PERM0=  {perm0},
 /
&DISSOLVED_GAS_PARAMETERS
 RHO0_D=  {rho0_d}     ,
 C0_D=  {c0_d}     ,
 CV_D=  {cv_d}     ,
 GAMMA_D=  {gamma_d}     ,
 P0_D=  {p0_d}     ,
 BAR_E_D=  {bar_e_d}     ,
 S0_D=  {s0_d}     ,
 EXSOL_MODEL="{exsol_model}",
 SOLUB=  {solub},
 SOLUB_EXP= {solub_exp}     ,
 /
&CRYSTALS_PARAMETERS
 RHO0_C=  {rho0_c}     ,
 C0_C=  {c0_c}     ,
 CV_C=  {cv_c}     ,
 GAMMA_C=  {gamma_c}     ,
 P0_C=  {p0_c}     ,
 BAR_E_C=  {bar_e_c}     ,
 S0_C=  {s0_c}     ,
 BETA0= {beta0}     ,
 BETA_MAX= {beta_max}    ,
 CRYSTALLIZATION_MODEL="{cryst_model}",
 /
&MELT_PARAMETERS
 RHO0_M=  {rho0_m}     ,
 C0_M=  {c0_m}     ,
 CV_M=  {cv_m}     ,
 GAMMA_M=  {gamma_m}     ,
 P0_M=  {p0_m}     ,
 BAR_E_M=  {bar_e_m}     ,
 S0_M=  {s0_m}     ,
 /
&VISCOSITY_PARAMETERS
 VISC_MELT_MODEL="{visc_melt_model}",
 BUBBLES_MODEL="{bubbles_model}",
 THETA_MODEL="{theta_model}",
 THETA_FIXED=  {theta_fixed}    ,
 /
&TEMPERATURE_PARAMETERS
 ISOTHERMAL={isothermal},
 FIXED_TEMP= {fixed_temp}     ,
 /
&FRAGMENTATION_PARAMETERS
 FRAGMENTATION_MODEL={fragmentation_mode}          ,
 FRAG_THR= {frag_thr}     ,
 /
&SOURCE_PARAMETERS
 GRAV=  {grav}    ,
 /
&RELAXATION_PARAMETERS
 DRAG_FUNCT_MODEL="{drag_funct_model}",
 LOG10_DRAG_FUNCT_COEFF=  0.0000000000000000     ,
 P_RELAX_MODEL="{p_relax_model}",
 LOG10_TAU_P_COEFF= {log10_tau_p_coeff}    ,
 LOG10_TAU_C= {log10_tau_c}     ,
 LOG10_TAU_D= {log10_tau_d}     ,
 /
&FORCHHEIMER_PARAMETERS
 LOG10_BUBBLE_NUMBER_DENSITY=  {log10_bubble_number_density}     ,
 TORTUOSITY_FACTOR=  {tortuosity_factor}    ,
 THROAT_BUBBLE_RATIO= {throat_bubble_ratio}    ,
 FRICTION_COEFFICIENT=  {friction_coefficient}     ,
 C_D= {c_d}     ,
 R_A=  {r_a},
 /
 """
out = open("conduit_solver.inp",'w')
out.write(inp)
out.close()

In [ ]:
subprocess.run("./MAMMA", shell=True)

#### Due to the file formatting of the outputs, please open Matlab on the desktop and run the `plot_solution.m` file for a detailed plot.